# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the high-level metadata object (as per mlcroissant's API)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

A Croissant dataset is organized in one or more record sets. We'll enumerate them, and for each record set, list their available fields and columns by their `@id`.

In [ ]:
# List all available record sets and their @id
for record_set in metadata.record_sets:
    print(f"Record Set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    # List all fields in this record set
    print("  Fields (by @id and name):")
    for field in record_set.fields:
        print(f"    - {field.id}: {field.name}")
    print("")

Inspecting a sample record from the main record set (by `@id`). You can pick the primary table for protein data, for instance, `cr:RecordSet/Proteins` (check above for the actual `@id` present in this dataset).

In [ ]:
# Let's pick the main record set @id. If unsure, see output above - e.g. 'cr:RecordSet/Proteins'.
primary_record_set_id = None
# Auto-selection of a record set if only one is present
if len(metadata.record_sets) == 1:
    primary_record_set_id = metadata.record_sets[0].id
    print(f"Automatically selected record set: {primary_record_set_id}")
else:
    # Fallback: manually check for the record set containing 'Protein' or similar
    for record_set in metadata.record_sets:
        if 'protein' in record_set.name.lower():
            primary_record_set_id = record_set.id
            print(f"Detected proteins record set: {primary_record_set_id}")
            break
    if not primary_record_set_id:
        print("Please set primary_record_set_id manually based on above list.")

if primary_record_set_id is not None:
    # Show a sample record
    for i, rec in enumerate(dataset.records(record_set=primary_record_set_id)):
        print(f"Sample record {i+1}:", rec)
        if i == 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s you discovered above.

In [ ]:
record_sets_ids = [rs.id for rs in metadata.record_sets]
# Create a dictionary of DataFrame for each record set
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()  # Empty DataFrame

# Use the primary record set for display (set above)
display_set_id = primary_record_set_id or (record_sets_ids[0] if record_sets_ids else None)
if display_set_id and not dataframes[display_set_id].empty:
    print(f"DataFrame columns from {display_set_id}:")
    print(dataframes[display_set_id].columns.tolist())

    print("\nFirst 5 rows:")
    display(dataframes[display_set_id].head())
else:
    print("No records found in the selected record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll try to select a numeric field (such as abundance, coverage, or molecular weight) for demonstration. Please use the field name (column) as listed in the DataFrame above.

In [ ]:
# Inspect all numeric columns in the chosen data frame
df = dataframes.get(display_set_id)
if df is not None and not df.empty:
    numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    print("Numeric fields:", numeric_cols)
    # Fallback: try to infer a numeric column by common mass spec names
    for col in df.columns:
        if any(term in col.lower() for term in ['abundance','coverage','count','mw','weight','area']) and col not in numeric_cols:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_cols.append(col)
            except Exception:
                pass
    print("(Revised) Numeric fields:", numeric_cols)
    
    # Pick the first numeric field found
    numeric_field = numeric_cols[0] if numeric_cols else None
    if numeric_field is not None:
        threshold = df[numeric_field].mean() if df[numeric_field].notna().sum() > 0 else 0
        
        print(f"Filtering records with {numeric_field} > {threshold:.2f}")
        filtered_df = df[df[numeric_field] > threshold]
        print(filtered_df.head())
        
        # Normalization
        warnings.filterwarnings('ignore')  # ignore SettingWithCopyWarning
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a relevant field (e.g. 'Sample' or 'Group' if available)
        candidate_group_fields = [c for c in df.columns if any(x in c.lower() for x in ['sample','group','condition'])]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field, as_index=False).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (mean of numeric):")
            print(grouped_df.head())
        else:
            print("No field found for grouping.")
    else:
        print("No numeric field detected for demonstration.")
else:
    print("No data in dataframe to explore.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the chosen numeric field and, if grouping is available, show group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_field is not None:
    # Plot histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field found, show a boxplot
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No suitable numeric field or data available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides quantitative proteomic data from extracellular vesicles isolated from human mast cells using mass spectrometry.
- We explored the record sets, loaded the main protein abundance table, filtered and normalized an example numeric field, and visualized its distribution.
- You can proceed with further analyses, such as protein annotation, clustering, or downstream bioinformatics, using the provided DataFrames.